### LangChain with Memory


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv('.env')
from langchain_core.messages import HumanMessage,SystemMessage

In [18]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_GPT_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000
)

In [21]:
from langchain_groq import ChatGroq

# FIXED: Use correct environment variable name and proper ChatGroq initialization
model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API"),  # FIXED: Use GROQ_API_KEY (matches .env file)
    max_tokens=1000
)

res = model.invoke([HumanMessage(content="Hello World!")])
print('Groq Response:', res.content)

Groq Response: Hello World! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [22]:
history=[]
while True:
    query=input("User: ")
    if query.lower() in ["exit","bye","quit"]:  
        print("GoodBye 👋")
        break
    
    history.append({"role":"user","content":query})
    print("User: ",query)

    res=llm.invoke(history)
    history.append({"role":"ai", "content":res.content})
    print("AI: ",res.content,"\n")

GoodBye 👋


In [23]:
# ALTERNATIVE: Using Groq with conversation memory
groq_history = []

print("🤖 Groq Chat with Memory (type 'exit' to quit)")
print("=" * 50)

while True:
    query = input("You: ")
    if query.lower() in ["exit", "bye", "quit"]:
        print("👋 Goodbye!")
        break

    # Add user message to history
    groq_history.append(HumanMessage(content=query))

    # Get Groq response with full conversation context
    groq_res = model.invoke(groq_history)

    # Add AI response to history
    groq_history.append(groq_res)

    print(f"Groq: {groq_res.content}")
    print("-" * 30)

🤖 Groq Chat with Memory (type 'exit' to quit)
👋 Goodbye!


### Streaming Technique
line-by-line printing

In [26]:
for chunk in model.stream("write a poem about a beautiful girl-boy in 1000 words"):
    print(chunk.content, end='', flush=True)

In a world of contrasts, where day meets night,
A beautiful girl-boy shone with radiant light.
Their name was Luna, a being of wonder and might,
A fusion of two worlds, where love and beauty took flight.

Their eyes, like sapphires, sparkled with a gentle hue,
Reflecting the stars on a clear and moonless night or two.
Their hair, a rich chestnut brown, cascaded down their back,
A cascade of silk, that seemed to whisper secrets to the wind's gentle crack.

Their skin, a porcelain doll's complexion, smooth and fair,
Inviting all to touch, to feel the gentle caress of their tender air.
Their lips, a rosebud's promise, inviting and alluring too,
A whispered promise of love, that only the heart could pursue.

Their body, a symphony of curves and lines,
A work of art, crafted by the divine.
Their smile, a ray of sunshine, that lit up the night,
A beacon of hope, that shone with all its might.

But Luna was more than just a pretty face,
Their soul, a deep and abiding well of love and kindness

### Batch Technique
parallel printing of each query

In [ ]:
responses=model.batch([
    "Who are u?",
    "In short DS..",
    "LPU University"],

    config={
        'max_concurrency':1 #Limits Parallel Requests to 2
    }
)
for response in responses:
    print(response.content)

[AIMessage(content="I'm an artificial intelligence model known as a large language model (LLM) or a conversational AI. I'm designed to understand and respond to human language, providing information, answering questions, and engaging in conversation.\n\nI don't have a personal identity or emotions, but I'm here to assist you with any questions or topics you'd like to discuss. I can provide information on a wide range of subjects, from science and history to entertainment and culture.\n\nI'm constantly learning and improving, so the more conversations I have, the more accurate and informative my responses become. I'm not perfect, but I'm here to help and provide assistance whenever you need it.\n\nWhat would you like to talk about?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 39, 'total_tokens': 182, 'completion_time': 0.51108833, 'completion_tokens_details': None, 'prompt_time': 0.008062205, 'prompt_tokens_details': None, 'queue_

In [37]:
model.invoke("Latest AI news of jan,2026")

AIMessage(content='Please note that my knowledge cutoff is December 2023, and I do not have real-time information. However, I can provide some general information and potential news trends that might have occurred in January 2026, based on my training data.\n\n**AI Trends and News (Potential) in January 2026:**\n\n1. **Advancements in Chatbots and Virtual Assistants**: The AI industry is expected to witness significant advancements in chatbots and virtual assistants, enabling more human-like interactions and personalization.\n2. **Increased Adoption of Edge AI**: As the demand for faster and more efficient AI processing grows, edge AI is expected to become more prevalent, enabling real-time processing and reducing latency.\n3. **Breakthroughs in Natural Language Processing (NLP)**: Researchers are working on improving NLP capabilities, enabling AI systems to better understand and generate human language.\n4. **Growing Concerns about AI Bias and Transparency**: As AI becomes more widesp

### Tools Integration

In [38]:
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get the current weather for a given location."""
    # Dummy implementation for illustration
    return f"The current weather in {location} is sunny with a temperature of 25°C."

In [39]:
get_weather("Mohali")

TypeError: 'StructuredTool' object is not callable